# 📈 Stock Price Prediction with Linear Regression
**Tools:** `yfinance` · `pandas` · `scikit-learn` · `matplotlib`

This notebook walks through every step: downloading data, feature engineering, training, evaluation, and visualisation.

## ⚙️ Step 1 — Install & import libraries

In [ ]:
# Install yfinance (not pre-installed on Colab)
%pip install yfinance --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('✅  All libraries loaded')

## 📥 Step 2 — Download historical stock data
Change `TICKER` to any valid symbol (e.g. `MSFT`, `TSLA`, `GOOGL`).

In [ ]:
TICKER = 'AAPL'   # ← change me
PERIOD = '2y'     # 2 years of daily data

print(f'📥  Downloading {TICKER} data...')
df = yf.download(TICKER, period=PERIOD, auto_adjust=True, progress=False)

# Newer yfinance returns MultiIndex columns — flatten them
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.dropna(inplace=True)
print(f'✅  {len(df)} rows downloaded  ({df.index[0].date()} → {df.index[-1].date()})')
display(df.tail())

## 🔧 Step 3 — Feature engineering
We derive predictive signals from raw prices: moving averages, momentum, volatility, and lag features.
The **target** is tomorrow's closing price (`Close` shifted by -1).

In [ ]:
# Target: next-day closing price
df['Target'] = df['Close'].shift(-1)

# Moving averages
df['SMA_5']  = df['Close'].rolling(window=5).mean()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()

# Momentum & volatility
df['Daily_Return']  = df['Close'].pct_change()
df['High_Low_Range'] = df['High'] - df['Low']
df['Volume_Change'] = df['Volume'] / df['Volume'].rolling(5).mean()

# Lag features
df['Lag_1'] = df['Close'].shift(1)
df['Lag_2'] = df['Close'].shift(2)

df.dropna(inplace=True)

print(f'✅  Features created. Dataset shape: {df.shape}')
display(df[['Close','SMA_5','EMA_10','Daily_Return','Lag_1','Target']].tail())

## ✂️ Step 4 — Train / test split
We keep time order intact — no shuffling. First 80% = training, last 20% = test.

In [ ]:
FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_5', 'SMA_10', 'SMA_20', 'EMA_10',
    'Daily_Return', 'High_Low_Range', 'Volume_Change',
    'Lag_1', 'Lag_2',
]

X = df[FEATURES].values
y = df['Target'].values.ravel()

SPLIT     = 0.80
split_idx = int(len(X) * SPLIT)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test      = df.index[split_idx:]

print(f'🔀  Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')

## ⚖️ Step 5 — Scale features
`StandardScaler` normalises each feature to mean=0, std=1.  
We fit **only on training data** to avoid leaking test-set statistics.

In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('✅  Features scaled')

## 🤖 Step 6 — Train the linear regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print('✅  Model trained!')

## 📊 Step 7 — Evaluate on test set

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

results = pd.DataFrame({
    'Metric': ['MAE (avg $ error)', 'RMSE (penalises big errors)', 'R² (1.0 = perfect)'],
    'Value' : [f'${mae:.2f}', f'${rmse:.2f}', f'{r2:.4f}']
})
display(results.style.hide(axis='index'))

## 🔮 Step 8 — Predict the next trading day's close

In [ ]:
last_row      = df[FEATURES].iloc[-1:].values
last_scaled   = scaler.transform(last_row)
next_day_pred = model.predict(last_scaled)[0]
last_close    = float(df['Close'].iloc[-1])
change        = next_day_pred - last_close
pct           = (change / last_close) * 100
arrow         = '▲' if change >= 0 else '▼'

print(f'📌  Last known close : ${last_close:.2f}')
print(f'🔮  Predicted close  : ${next_day_pred:.2f}  {arrow} {pct:+.2f}%')

## 📉 Step 9 — Visualise: actual vs predicted + residuals

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle(f'{TICKER} — Linear Regression Price Prediction', fontsize=15, fontweight='bold')

# --- Plot 1: Actual vs Predicted ---
ax1 = axes[0]
ax1.plot(dates_test, y_test, label='Actual next-day close',    color='#2196F3', linewidth=1.8)
ax1.plot(dates_test, y_pred, label='Predicted next-day close', color='#FF5722',
         linewidth=1.4, linestyle='--', alpha=0.85)
ax1.set_title('Actual vs Predicted — Test Period')
ax1.set_ylabel('Price (USD)')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax1.grid(alpha=0.3)

# --- Plot 2: Residuals ---
residuals = y_test - y_pred
ax2 = axes[1]
colors = ['#4CAF50' if r >= 0 else '#F44336' for r in residuals]
ax2.bar(dates_test, residuals, color=colors, alpha=0.7, width=1.5)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Residuals (Actual − Predicted)')
ax2.set_ylabel('Error (USD)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('stock_prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Chart saved as stock_prediction_results.png')

## 💾 (Optional) Download the chart to your computer

In [ ]:
from google.colab import files
files.download('stock_prediction_results.png')